In [1]:
import numpy as np
import pandas as pd
import talib
import sys
sys.path.append('.')

import math
from datetime import datetime, timedelta, date

pair_trades_filename = 'pairtrades.csv'
dateparse = lambda x: pd.datetime.strptime(x, '%Y-%m-%d')

df = pd.read_csv(pair_trades_filename, parse_dates=['create_date', 'active_date'])

beta_symbol = 'SPY'

position_limit = 20000

num_days_daily = 20
num_days_weekly = 30
num_days_monthly = 500

start_date_daily = date.today() - timedelta(days = math.ceil((num_days_daily/(5/7))))
start_date_weekly = date.today() - timedelta(days = math.ceil((num_days_weekly/(5/7))))
start_date_monthly = date.today() - timedelta(days = math.ceil((num_days_monthly/(5/7))))

df['create_date'] = df['create_date'].dt.date

df


,long,short,create_date,active_date,active_start_long,active_start_short,notes
0,IWM,QQQ,2024-11-30,NaT,NaN,NaN,NaN


In [4]:
from polygon import RESTClient
from dateutil import tz
def aggregates_to_df(resp):
    epoch = datetime(1601, 1, 1)
    df = pd.DataFrame(resp.results)
#     tz_ = tz.gettz('America/New_York')
    tz_ = tz.gettz('UTC')

    df.set_index(df['t'].apply(lambda x: datetime.fromtimestamp(x / 1000, tz=tz_)), inplace=True)
    df.rename(columns={"v": "volume", "o": "open", "c": "close", "h": "high", "l": "low"}, inplace=True)
    if 't' in df:
        df.drop(['t'], axis=1, inplace=True)
    if 'n' in df:
        df.drop(['n'], axis=1, inplace=True)

    df.index.name = None

    return df

def history(symbol, start_date, timespan = 'day'):
    date_format = "%Y-%m-%d"
    
    end_date = date.today() + timedelta(days = 7) # Polygon is exclusive on dates and must extend for a week
    client = RESTClient('WeYC_RTF_nZg3S_UKGI68lvNZ__8l6YM09p_Rg')
    multiplier = 1

    try:
        resp = client.stocks_equities_aggregates(symbol, multiplier, timespan, start_date.strftime(date_format), end_date.strftime(date_format))
    except Exception as e:
        print("[history_aggregate()]: Polygon connection error: %s" % (str(e)))

    if resp.status != 'OK':
        raise ValueError("Polygon history_aggregate response not OK: %s" % (resp['status']))
        
    if resp.results == None or len(resp.results) == 0:
        raise ValueError("Polygon history_aggregate results are empty")
        
    df = aggregates_to_df(resp)
    
    return df



In [5]:
import plotly.graph_objects as go
from statistics import mean 


for index, pair in df.iterrows():
    print("%s / %s" % (pair['long'], pair['short']))
    start_daily = min(start_date_daily, pair['create_date']) - timedelta(days = 7)
    start_weekly = min(start_date_weekly, pair['create_date']) - timedelta(days = 7)
    start_monthly = min(start_date_monthly, pair['create_date']) - timedelta(days = 7)

    beta_ohlcv = history(beta_symbol, start_daily)
    
    print("Create Date:\t%s" % (start_daily))
    print("")
    
    # Daily
    ohlcv_daily_long = history(pair['long'], start_daily)
    ohlcv_daily_short = history(pair['short'], start_daily)

    # Weekly
    ohlcv_week_long = history(pair['long'], start_weekly, timespan = 'week')
    ohlcv_week_short = history(pair['short'], start_weekly, timespan = 'week')

    # Monthly
    ohlcv_month_long = history(pair['long'], start_monthly, timespan = 'month')
    ohlcv_month_short = history(pair['short'], start_monthly, timespan = 'month')
    
    corrs = ohlcv_daily_long.corrwith(ohlcv_daily_short, axis=0)
    ratios = ohlcv_daily_long / ohlcv_daily_short
    ratios_week = ohlcv_week_long / ohlcv_week_short

    # ATR Daily
    atr_daily_long = talib.ATR(ohlcv_daily_long['high'].values, ohlcv_daily_long['low'].values, ohlcv_daily_long['close'].values, timeperiod=5)
    atr_daily_short = talib.ATR(ohlcv_daily_short['high'].values, ohlcv_daily_short['low'].values, ohlcv_daily_short['close'].values, timeperiod=5)

    atrp_daily_long = atr_daily_long / ohlcv_daily_long['close'].values
    atrp_daily_short = atr_daily_short / ohlcv_daily_short['close'].values

    # ATR Weekly
    atr_week_long = talib.ATR(ohlcv_week_long['high'].values, ohlcv_week_long['low'].values, ohlcv_week_long['close'].values, timeperiod=5)
    atr_week_short = talib.ATR(ohlcv_week_short['high'].values, ohlcv_week_short['low'].values, ohlcv_week_short['close'].values, timeperiod=5)

    atrp_week_long = atr_week_long / ohlcv_week_long['close'].values
    atrp_week_short = atr_week_short / ohlcv_week_short['close'].values

    # ATR Monthly
    atr_month_long = talib.ATR(ohlcv_month_long['high'].values, ohlcv_month_long['low'].values, ohlcv_month_long['close'].values, timeperiod=5)
    atr_month_short = talib.ATR(ohlcv_month_short['high'].values, ohlcv_month_short['low'].values, ohlcv_month_short['close'].values, timeperiod=5)

    atrp_month_long = atr_month_long / ohlcv_month_long['close'].values
    atrp_month_short = atr_month_short / ohlcv_month_short['close'].values

    # ATR Daily Pair
    atr_daily_pair = talib.ATR(ratios['high'].values, ratios['low'].values, ratios['close'].values, timeperiod=5)
    atrp_daily_pair = atr_daily_pair / ratios['close'].values
    
    # ATR Weekly Pair
    atr_weekly_pair = talib.ATR(ratios_week['high'].values, ratios_week['low'].values, ratios_week['close'].values, timeperiod=5)
    atrp_weekly_pair = atr_weekly_pair / ratios_week['close'].values
    
    # Beta
    beta_long = talib.BETA(beta_ohlcv['close'], ohlcv_daily_long['close'])[-1]
    beta_short = talib.BETA(beta_ohlcv['close'], ohlcv_daily_short['close'])[-1]
    beta_total = beta_long + beta_short
    beta_ratio = beta_long / beta_short
    
    # Position Sizing
    beta_long_dollars = (position_limit) * (1 - (beta_long / beta_total))
    beta_short_dollars = (position_limit) * (1 - (beta_short / beta_total))

    shares_long = math.floor(beta_long_dollars / ohlcv_daily_long.iloc[-1]['close'])
    shares_short = math.floor(beta_short_dollars / ohlcv_daily_short.iloc[-1]['close'])

    # Returns
    beta_returns_start = (ohlcv_daily_long.iloc[0]['close'] * shares_long) - (ohlcv_daily_short.iloc[0]['close'] * shares_short)
    beta_returns_end = (ohlcv_daily_long.iloc[-1]['close'] * shares_long) - (ohlcv_daily_short.iloc[-1]['close'] * shares_short)
    beta_returns_pnl = beta_returns_end - beta_returns_start
    beta_returns_pnl_pct = beta_returns_pnl / beta_returns_start
    
    print(str(ohlcv_daily_long.iloc[-1]['close']))
    print(str(ohlcv_daily_short.iloc[-1]['close']))
    print(str(shares_long))
    print(str(shares_short))
    
    # Print
    if not pd.isnull(pair['active_date']):
        ratio_start = pair['active_start_long'] / pair['active_start_short']
        ratio_current = ratios['close'][-1]
        
        stop_amount = mean([atrp_month_long[-1], atrp_month_short[-1]]) * ratio_start
        stop_loss = ratio_start - stop_amount
        soft_target = (3.0 * stop_amount) + ratio_start
        
        
        print("Ratio Start: \t%3.2f Current: \t%3.2f" % (ratio_start, ratio_current))
        print("PnL %%:\t\t%3.2f" % (((ratio_current - ratio_start) / ratio_start) * 100.0))
        print("Soft Target:\t%3.2f" % (soft_target))
        print("Stop Loss:\t%3.2f" % (stop_loss))


    print("ATR Daily:  \t%3.1f%%\t::: \t%3.1f%%\t: %3.1f%%" % (atrp_daily_pair[-1]*100.0, atrp_daily_long[-1]*100.0, atrp_daily_short[-1]*100.0))
    print("ATR Weekly: \t%3.1f%%\t::: \t%3.1f%%\t:: %3.1f%%" % (atrp_weekly_pair[-1]*100.0, atrp_week_long[-1]*100.0, atrp_week_short[-1]*100.0))
    print("ATR Monthly:\t\t\t%3.1f%%\t:: %3.1f%%" % (atrp_month_long[-1]*100.0, atrp_month_short[-1]*100.0))
    print("")
    print("Price:       \t%0.2f\t:: %0.2f" % (ohlcv_daily_long['close'].iloc[-1], ohlcv_daily_short['close'].iloc[-1]))
    print("Ratio:       \t%0.2f" % (ratios['close'][-1]))
    print("Correl:      \t%0.2f" % (corrs['close']))
    print("Beta Ratio:  \t%0.2f" % (beta_ratio))
    print("Beta:        \t%0.2f\t:: %0.2f" % (beta_long, beta_short))
    print("Long $$$:    \t%0.2f" % (beta_long_dollars))
    print("Short $$$:   \t%0.2f" % (beta_short_dollars))
    print("Long Shares: \t%d" % (shares_long))
    print("Short Shares:\t%d" % (shares_short))

    print("Returns:")
    print("Unhedged:    \t%0.2f%%" % (((ratios['close'][-1] - ratios['close'][0])/ratios['close'][0])*100.0))
    print("Beta Hedged: \t%0.2f%%" % (beta_returns_pnl_pct))
    
    print(str(ratios_week['close'].pct_change().dropna()*100.0))
#     print("Ratio Start: %0.2f" % (ratio_start))
#     print("Ratio End. : %0.2f" % (ratio_end))
    fig = go.Figure(data=go.Candlestick(
                    open=ratios['open'],
                    high=ratios['high'],
                    low=ratios['low'],
                    close=ratios['close']))
    fig.update_layout(title=('%s / %s' % (pair['long'], pair['short'])), yaxis_title='Ratio', xaxis_title='Day')
    fig.update(layout_xaxis_rangeslider_visible=False)
    fig.show()
    
    
    

IWM / QQQ
Create Date:	2024-11-13

232.2
534.43
44
18
ATR Daily:  	0.9%	::: 	1.3%	: 1.0%
ATR Weekly: 	2.3%	::: 	4.4%	:: 2.9%
ATR Monthly:			8.1%	:: 6.7%

Price:       	232.20	:: 534.43
Ratio:       	0.43
Correl:      	0.07
Beta Ratio:  	0.95
Beta:        	2.16	:: 2.26
Long $$$:    	10232.34
Short $$$:   	9767.66
Long Shares: 	44
Short Shares:	18
Returns:
Unhedged:    	-5.34%
Beta Hedged: 	-0.47%
2024-11-03 04:00:00+00:00    3.091364
2024-11-10 05:00:00+00:00   -0.653349
2024-11-17 05:00:00+00:00    2.598688
2024-11-24 05:00:00+00:00    0.513355
2024-12-01 05:00:00+00:00   -4.356488
2024-12-08 05:00:00+00:00   -3.197266
2024-12-15 05:00:00+00:00   -1.100304
Name: close, dtype: float64
